# 01 · Profiling the English Prescribing Dataset

**Purpose:** understand what the raw data *means* before writing any cleaning or business logic.
Every finding here became a rule in the dbt staging model (`dbt/models/staging/stg_epd.sql`) or a design decision for the marts.

**Data:** NHSBSA EPD (SNOMED edition), May–July 2026, landed as Parquet by `ingest/epd.py`.

**Approach:** DuckDB queries straight over the Parquet files. Results come back as pandas DataFrames for display.
No cleaning happens here: this notebook explores, and dbt cleans.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 20)
con = duckdb.connect()

# hive_partitioning=false: see finding 0
con.execute("""
    create view epd as
    select * from read_parquet('../data/raw/epd/*/epd.parquet', hive_partitioning=false)
""")
q = lambda sql: con.sql(sql).df()

## 0 · Gotcha: the folder name overwrites a column

Files live in hive-style folders (`year_month=202607/`). DuckDB auto-detects this and adds a `year_month` column.
Column names are case-insensitive, so it **silently replaces** the file's own `YEAR_MONTH` value.

In [2]:
q("""
select
    (select any_value(YEAR_MONTH) from read_parquet('../data/raw/epd/year_month=202607/epd.parquet'))
        as default_read,
    (select any_value(YEAR_MONTH) from read_parquet('../data/raw/epd/year_month=202607/epd.parquet', hive_partitioning=false))
        as hive_off
""")

,default_read,hive_off
0,202607,2026-07


**Rule:** always read with `hive_partitioning=false`. The staging source does this.

## 1 · Volume

In [3]:
q("""
select YEAR_MONTH                          as month,
       count(*)                            as n_rows,
       count(distinct PRACTICE_CODE)       as practices,
       count(distinct ICB_CODE)            as icbs,
       count(distinct BNF_PRESENTATION_CODE) as presentations,
       sum(ITEMS)::bigint                  as items,
       round(sum(NIC) / 1e6, 1)            as nic_gbp_m,
       round(sum(ACTUAL_COST) / 1e6, 1)    as actual_cost_gbp_m
from epd group by 1 order by 1
""")

,month,n_rows,practices,icbs,presentations,items,nic_gbp_m,actual_cost_gbp_m
0,2026-05,18000093,9278,37,21275,104152360,938.1,900.1
1,2026-06,18374449,9275,37,21404,109213199,997.3,957.5
2,2026-07,18601776,9284,37,21422,113369726,1044.6,1000.6


About 18M rows and ~£1bn of net ingredient cost (NIC) each month, across ~9.3k practices and 37 ICBs.
The row counts match the portal's published totals, so ingest lost nothing.

## 2 · Grain: what is one row?

The obvious guess is practice × drug (BNF presentation) × SNOMED code × month. Let's test it.

In [4]:
q("""
select count(*) as duplicated_key_groups
from (select 1 from epd
      where YEAR_MONTH = '2026-07'
      group by PRACTICE_CODE, BNF_PRESENTATION_CODE, SNOMED_CODE
      having count(*) > 1)
""")

,duplicated_key_groups
0,4007316


Millions of duplicates, so the guess is wrong. Look at one example:

In [5]:
q("""
select PRACTICE_CODE, BNF_PRESENTATION_NAME, QUANTITY, ITEMS, TOTAL_QUANTITY, NIC
from epd
where YEAR_MONTH = '2026-07' and PRACTICE_CODE = 'E85107' and BNF_PRESENTATION_CODE = '0401040A0AAAAAA'
""")

,PRACTICE_CODE,BNF_PRESENTATION_NAME,QUANTITY,ITEMS,TOTAL_QUANTITY,NIC
0,E85107,Daridorexant 25mg tablets,60.0,2.0,120.0,168.0
1,E85107,Daridorexant 25mg tablets,30.0,4.0,120.0,168.0


**Finding:** the source splits rows by prescription size. `QUANTITY` is **per item**, and
`TOTAL_QUANTITY = ITEMS × QUANTITY` is the real volume. The same 120 tablets arrive as 2 items × 60 or 4 items × 30.

Check that identity holds everywhere:

In [6]:
q("""
select count(*) filter (where abs(TOTAL_QUANTITY - ITEMS * QUANTITY) > 0.01) as rows_breaking_identity,
       count(*) as n_rows
from epd
""")

,rows_breaking_identity,n_rows
0,0,54976318


Add quantity-per-item to the key and re-test:

In [7]:
q("""
select PRACTICE_CODE = '-' as is_unidentified, count(*) as duplicated_key_groups
from (select PRACTICE_CODE from epd
      where YEAR_MONTH = '2026-07'
      group by PRACTICE_CODE, BNF_PRESENTATION_CODE, SNOMED_CODE, QUANTITY
      having count(*) > 1)
group by 1
""")

,is_unidentified,duplicated_key_groups
0,True,63


The only duplicates left are **unidentified prescribing** (`PRACTICE_CODE = '-'`), where prescriptions from unknown prescribers are pooled.

**Rules:**
- The grain is practice × presentation × SNOMED × quantity-per-item × month, and it isn't unique by design.
- Staging renames `QUANTITY` → `quantity_per_item` so nobody treats it as a total.
- Unit prices use `nic / total_quantity`. Using `QUANTITY` would inflate them by the number of items.
- No `unique` test on staging. Uniqueness is tested on marts, whose grain we control by aggregating.

## 3 · BNF code shapes

In [8]:
q("""
select length(BNF_PRESENTATION_CODE) as code_length,
       left(BNF_CHAPTER_PLUS_CODE, 40) as chapter,
       count(*) as n_rows,
       round(sum(NIC) / 1e6, 1) as nic_gbp_m
from epd where YEAR_MONTH = '2026-07'
group by all order by code_length, n_rows desc
""")

,code_length,chapter,n_rows,nic_gbp_m
0,11,21: Appliances,1181624,88.0
1,11,23: Stoma Appliances,359375,40.0
2,11,20: Dressings,271279,15.8
3,11,22: Incontinence Appliances,120282,6.2
4,15,04: Central Nervous System,4619485,153.4
5,15,02: Cardiovascular System,2770610,130.3
6,15,06: Endocrine System,1922558,222.3
7,15,09: Nutrition and Blood,1367757,80.1
8,15,01: Gastro-Intestinal System,1341554,63.8
9,15,03: Respiratory System,974673,112.1


**Finding:** drugs (chapters 01–19) have 15-character codes. Appliances, dressings, stoma and incontinence products
(chapters 20–23, ~£150M/month) have **11-character** codes with no product/generic structure.

**Rule:** `is_drug = length = 15 and chapter <= '19'`. Generic flags are derived only for drugs, and are NULL otherwise.

## 4 · Nulls, negatives, zeros, unidentified

In [9]:
q("""
select count(*) filter (where NIC is null)            as null_nic,
       count(*) filter (where ITEMS is null)          as null_items,
       count(*) filter (where NIC < 0)                as negative_nic,
       count(*) filter (where NIC = 0)                as zero_nic,
       count(*) filter (where UNIDENTIFIED = 'Y')     as unidentified_rows,
       round(sum(NIC) filter (where UNIDENTIFIED = 'Y') / 1e6, 2) as unidentified_nic_gbp_m
from epd
""")

,null_nic,null_items,negative_nic,zero_nic,unidentified_rows,unidentified_nic_gbp_m
0,0,0,0,1262,47674,1.88


Clean on these checks: no nulls and no negative costs. The zero-cost rows are valid (free-of-charge items), and unidentified prescribing is ~0.06% of spend.

**Rule:** keep unidentified rows, flagged, so national and ICB totals still reconcile.

## 5 · Generic vs branded, and the savings question

BNF code characters 10–11 are the product: `AA` means generic. The generic equivalent of any drug is
`chars 1–9 + 'AA' + chars 14–15 + chars 14–15`, the same rule OpenPrescribing uses.

In [10]:
q("""
with d as (
    select *,
           substr(BNF_PRESENTATION_CODE, 10, 2) = 'AA' as is_generic,
           substr(BNF_PRESENTATION_CODE, 1, 9) || 'AA'
             || substr(BNF_PRESENTATION_CODE, 14, 2) || substr(BNF_PRESENTATION_CODE, 14, 2) as generic_equiv
    from epd
    where YEAR_MONTH = '2026-07' and length(BNF_PRESENTATION_CODE) = 15 and left(BNF_PRESENTATION_CODE, 2) <= '19'
),
g as (select distinct BNF_PRESENTATION_CODE from d where is_generic)
select left(BNF_CHAPTER_PLUS_CODE, 35) as chapter,
       round(sum(NIC) filter (where is_generic) / 1e6, 1)     as generic_gbp_m,
       round(sum(NIC) filter (where not is_generic) / 1e6, 1) as branded_gbp_m,
       round(sum(NIC) filter (where not is_generic and generic_equiv in (select * from g)) / 1e6, 1)
           as branded_with_generic_gbp_m
from d group by 1 order by branded_with_generic_gbp_m desc limit 8
""")

,chapter,generic_gbp_m,branded_gbp_m,branded_with_generic_gbp_m
0,06: Endocrine System,88.3,134.1,121.9
1,03: Respiratory System,13.6,98.5,90.1
2,04: Central Nervous System,117.5,35.9,35.8
3,01: Gastro-Intestinal System,39.5,24.3,23.3
4,09: Nutrition and Blood,14.0,66.0,11.1
5,02: Cardiovascular System,120.1,10.3,10.0
6,08: Malignant Disease and Immunosup,6.1,9.6,9.6
7,13: Skin,15.3,12.0,8.8


**Finding:** ~80% of branded drug spend has an equivalent generic that is also prescribed. It's concentrated in
endocrine (insulins and similar) and respiratory (inhalers), where brand prescribing is often *clinically correct* or
the brand costs the same.

**Design consequences for the marts:**
- Savings = the actual price difference against a reference generic price, not "all branded spend".
- A `brand_exceptions` list is needed for drugs that should be prescribed by brand.

## 6 · Validating the low-value medicines seed

`dbt/seeds/low_value_medicines.csv` maps NHS England's "should not routinely be prescribed" list to BNF `LIKE` patterns.
Fentanyl immediate-release is matched on the generic-equivalent suffix (`0407020A0%AW`), which also catches brands.
The risk is catching patches or injections, which are not on the list:

In [11]:
con.execute("create view lv as select * from read_csv('../dbt/seeds/low_value_medicines.csv', all_varchar=true)")
q("""
select distinct e.BNF_PRESENTATION_NAME
from epd e join lv on e.BNF_PRESENTATION_CODE like lv.bnf_code_like
where lv.category like 'fentanyl%'
  and regexp_matches(lower(e.BNF_PRESENTATION_NAME), 'patch|inject|infus')
""")

,BNF_PRESENTATION_NAME


No false matches. Spend by category (rules marked `is_exclusion` are subtracted, e.g. topical NSAIDs among the rubefacients):

In [12]:
q("""
select lv.category, e.YEAR_MONTH as month, round(sum(e.NIC) / 1e3, 1) as nic_gbp_k
from epd e join lv on e.BNF_PRESENTATION_CODE like lv.bnf_code_like and lv.is_exclusion = 'false'
where not exists (select 1 from lv x where x.is_exclusion = 'true' and e.BNF_PRESENTATION_CODE like x.bnf_code_like)
group by all
""").pivot(index="category", columns="month", values="nic_gbp_k").sort_values("2026-07", ascending=False)

month,2026-05,2026-06,2026-07
category,,,
lidocaine plasters,1193.2,1252.0,1315.3
liothyronine,708.6,757.6,775.3
fentanyl immediate release,335.2,333.6,342.6
dosulepin,270.3,274.3,239.7
omega-3 fatty acids,183.9,188.6,200.2
amiodarone,278.9,286.8,186.4
co-proxamol,160.2,174.7,159.4
trimipramine,139.0,132.6,135.3
oxycodone + naloxone,99.3,95.0,108.2


## Summary: findings → rules

| # | Finding | Where it's handled |
|---|---|---|
| 0 | The folder name overwrites `YEAR_MONTH` | `hive_partitioning=false` in the dbt source |
| 2 | Rows are split by prescription size; `QUANTITY` is per item | `quantity_per_item` rename; unit price uses `total_quantity`; no `unique` test on staging |
| 2 | The only leftover duplicates are pooled unidentified prescribing | Kept and flagged `is_unidentified` |
| 3 | Appliances use 11-character codes | `is_drug` flag; generic fields NULL for non-drugs |
| 4 | No nulls or negative costs | No cleaning needed; `not_null` tests guard it |
| 5 | 80% of branded spend has a prescribed generic | Savings by price difference, plus a brand-exceptions list (phase 3) |
| 6 | Low-value rules are validated | Seed used as-is in `mart_low_value` |